# Data Cleaning Pipeline - Market Mix Modeling

This notebook implements a comprehensive data cleaning pipeline for the MMM dataset with:
1. Missing value treatment
2. Duplicate removal
3. Datatype conversion
4. Outlier detection
5. Data validation checks
6. Save cleaned dataset

In [3]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Load Raw Data

In [4]:
# Load the raw dataset
data_path = '../data/raw/synthetic_mmm_weekly_india.csv'
df_raw = pd.read_csv(data_path)

# Create working copy
df = df_raw.copy()

print(f"✓ Raw dataset loaded: {data_path}")
print(f"\nInitial Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumn Names: {list(df.columns)}")

✓ Raw dataset loaded: ../data/raw/synthetic_mmm_weekly_india.csv

Initial Dataset Shape: 11,232 rows × 28 columns
Memory Usage: 4.52 MB

Column Names: ['Week', 'Geo', 'Brand', 'SKU', 'Sales_Units', 'Sales_Value', 'MRP', 'Net_Price', 'Feature_Flag', 'Display_Flag', 'TPR_Flag', 'Trade_Spend', 'TV_Impressions', 'YouTube_Impressions', 'Facebook_Impressions', 'Instagram_Impressions', 'Print_Readership', 'Radio_Listenership', 'FB_Banner_Content_Score', 'IG_Banner_Content_Score', 'Weighted_Distribution', 'Numeric_Distribution', 'TDP', 'NOS', 'CPI', 'GDP_Growth', 'Festival_Index', 'Rainfall_Index']


## 2. Missing Value Treatment

In [5]:
print("="*80)
print("STEP 1: MISSING VALUE ANALYSIS & TREATMENT")
print("="*80)

# Check missing values
missing_before = df.isnull().sum()
missing_pct = (missing_before / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_before.values,
    'Missing_Percentage': missing_pct.values,
    'Data_Type': df.dtypes.values
})

missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_summary) > 0:
    print("\nColumns with Missing Values:")
    print(missing_summary.to_string(index=False))
else:
    print("\n✓ No missing values detected in the dataset!")

total_missing_before = missing_before.sum()
print(f"\nTotal Missing Values: {total_missing_before}")
print(f"Data Completeness: {100 - (total_missing_before / (df.shape[0] * df.shape[1]) * 100):.2f}%")

# Treatment Strategy
print("\n" + "-"*80)
print("Treatment Strategy:")
print("-"*80)

# For categorical columns: fill with mode or 'Unknown'
categorical_cols = [col for col in df.columns if df[col].dtype == 'object']
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col].fillna(mode_val, inplace=True)
        print(f"  → {col}: Filled with mode: '{mode_val}'")

# For numerical columns: fill with median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols[:10]}... (showing first 10)")

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  → {col}: Filled with median: {median_val:.2f}")

total_missing_after = df.isnull().sum().sum()
print(f"\n✓ Missing Value Treatment Complete")
print(f"  Before: {total_missing_before} missing values")
print(f"  After: {total_missing_after} missing values")

STEP 1: MISSING VALUE ANALYSIS & TREATMENT

✓ No missing values detected in the dataset!

Total Missing Values: 0
Data Completeness: 100.00%

--------------------------------------------------------------------------------
Treatment Strategy:
--------------------------------------------------------------------------------

Categorical columns (0): []

Numeric columns (24): ['Sales_Units', 'Sales_Value', 'MRP', 'Net_Price', 'Feature_Flag', 'Display_Flag', 'TPR_Flag', 'Trade_Spend', 'TV_Impressions', 'YouTube_Impressions']... (showing first 10)

✓ Missing Value Treatment Complete
  Before: 0 missing values
  After: 0 missing values


## 3. Duplicate Removal

In [6]:
print("="*80)
print("STEP 2: DUPLICATE REMOVAL")
print("="*80)

# Check for complete duplicates
complete_duplicates = df.duplicated().sum()
print(f"\nComplete Duplicates (all columns): {complete_duplicates}")
print(f"Percentage: {complete_duplicates / len(df) * 100:.2f}%")

# Check for duplicates on key columns
if 'Week' in df.columns and 'Geo' in df.columns and 'Brand' in df.columns and 'SKU' in df.columns:
    key_cols = ['Week', 'Geo', 'Brand', 'SKU']
    key_duplicates = df.duplicated(subset=key_cols).sum()
    print(f"\nDuplicates on Key Columns {key_cols}: {key_duplicates}")
    print(f"Percentage: {key_duplicates / len(df) * 100:.2f}%")

# Remove duplicates
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"\n✓ Duplicate Removal Complete")
print(f"  Rows before: {rows_before:,}")
print(f"  Rows after: {rows_after:,}")
print(f"  Rows removed: {rows_removed:,}")

# Verify no duplicates remain
remaining_duplicates = df.duplicated().sum()
print(f"\n  Remaining duplicates: {remaining_duplicates}")

STEP 2: DUPLICATE REMOVAL

Complete Duplicates (all columns): 0
Percentage: 0.00%

Duplicates on Key Columns ['Week', 'Geo', 'Brand', 'SKU']: 0
Percentage: 0.00%

✓ Duplicate Removal Complete
  Rows before: 11,232
  Rows after: 11,232
  Rows removed: 0

  Remaining duplicates: 0


## 4. Datatype Conversion

In [7]:
print("="*80)
print("STEP 3: DATATYPE CONVERSION")
print("="*80)

print("\nCurrent Data Types:")
print(df.dtypes)

# Define optimal datatypes
print("\n" + "-"*80)
print("Optimizing Data Types for Memory Efficiency:")
print("-"*80)

# Convert object columns to category where appropriate
categorical_candidates = ['Week', 'Geo', 'Brand', 'SKU']
for col in categorical_candidates:
    if col in df.columns and df[col].dtype == 'object':
        unique_count = df[col].nunique()
        if unique_count < len(df) * 0.05:  # If less than 5% unique values
            df[col] = df[col].astype('category')
            print(f"  → {col}: object → category (Unique values: {unique_count})")

# Convert integer columns to int32 if possible (memory optimization)
int_cols = df.select_dtypes(include=['int64']).columns.tolist()
for col in int_cols:
    if df[col].min() >= 0 and df[col].max() < 2**31:  # Can fit in int32
        df[col] = df[col].astype('int32')
        print(f"  → {col}: int64 → int32")

# Convert float64 to float32 for non-critical columns (memory optimization)
float_cols = df.select_dtypes(include=['float64']).columns.tolist()
for col in float_cols:
    df[col] = df[col].astype('float32')
    print(f"  → {col}: float64 → float32")

# Flag columns conversion (0/1 to bool for efficiency)
flag_cols = [col for col in df.columns if 'Flag' in col]
for col in flag_cols:
    if set(df[col].unique()) <= {0, 1, 0.0, 1.0}:
        df[col] = df[col].astype('bool')
        print(f"  → {col}: numeric → bool")

print("\n✓ Datatype Conversion Complete")

print("\nOptimized Data Types:")
print(df.dtypes)

# Memory comparison
memory_before = df_raw.memory_usage(deep=True).sum() / 1024**2
memory_after = df.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
memory_saved_pct = (memory_saved / memory_before) * 100

print(f"\nMemory Usage Comparison:")
print(f"  Before: {memory_before:.2f} MB")
print(f"  After: {memory_after:.2f} MB")
print(f"  Saved: {memory_saved:.2f} MB ({memory_saved_pct:.1f}%)")

STEP 3: DATATYPE CONVERSION

Current Data Types:
Week                           str
Geo                            str
Brand                          str
SKU                            str
Sales_Units                float64
Sales_Value                float64
MRP                        float64
Net_Price                  float64
Feature_Flag                 int64
Display_Flag                 int64
TPR_Flag                     int64
Trade_Spend                float64
TV_Impressions             float64
YouTube_Impressions        float64
Facebook_Impressions       float64
Instagram_Impressions      float64
Print_Readership           float64
Radio_Listenership         float64
FB_Banner_Content_Score    float64
IG_Banner_Content_Score    float64
Weighted_Distribution      float64
Numeric_Distribution       float64
TDP                        float64
NOS                        float64
CPI                        float64
GDP_Growth                 float64
Festival_Index             float64
Rainfa

## 4. Outlier Detection and Treatment
Detect and cap outliers using the IQR rule on continuous numeric columns.

In [ ]:
print("="*80)
print("STEP 4: OUTLIER DETECTION AND TREATMENT")
print("="*80)

# Detect and cap outliers using the IQR rule on continuous numeric columns
flag_cols = [col for col in df.columns if 'Flag' in col]
continuous_numeric_cols = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if col not in flag_cols
]

outlier_summary = []
for col in continuous_numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outlier_count = int(outlier_mask.sum())

    if outlier_count > 0:
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    outlier_summary.append({
        'Column': col,
        'Outlier_Count': outlier_count,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound
    })

outlier_summary_df = pd.DataFrame(outlier_summary).sort_values('Outlier_Count', ascending=False)
columns_with_outliers = outlier_summary_df[outlier_summary_df['Outlier_Count'] > 0]

if len(columns_with_outliers) > 0:
    print("\nColumns with detected outliers:")
    print(columns_with_outliers.to_string(index=False))
else:
    print("\n✓ No outliers detected using the IQR rule")

print("\n✓ Outlier treatment complete: values outside the IQR bounds were capped")

STEP 4: OUTLIER DETECTION AND TREATMENT

Columns with detected outliers:
               Column  Outlier_Count    Lower_Bound  Upper_Bound
     Print_Readership            888  -40680.552734 1.837647e+05
       TV_Impressions            698 -880078.820312 2.651426e+06
   Radio_Listenership            692 -249230.326172 7.956672e+05
          Sales_Units            679    -286.949579 1.022168e+03
          Sales_Value            673  -25274.925171 8.214578e+04
 Facebook_Impressions            638 -310379.562500 8.580286e+05
          Trade_Spend            603  -86405.187744 1.778058e+05
  YouTube_Impressions            586 -478819.425781 1.411987e+06
Instagram_Impressions            547  -88848.494141 3.371588e+05
                  NOS             63       2.524108 6.288344e+00
           GDP_Growth             36       3.514926 6.351522e+00

✓ Outlier treatment complete: values outside the IQR bounds were capped

STEP 5: DATA VALIDATION CHECKS
                        Check  Passed     

## 5. Data Validation Checks
Validate key business rules and data quality constraints.

In [9]:
print("="*80)
print("STEP 5: DATA VALIDATION CHECKS")
print("="*80)

validation_results = []

def add_validation_check(check_name, passed, details):
    validation_results.append({
        'Check': check_name,
        'Passed': passed,
        'Details': details
    })

required_columns = [
    'Week', 'Geo', 'Brand', 'SKU', 'Sales_Units', 'Sales_Value', 'MRP', 'Net_Price'
]
missing_required = [col for col in required_columns if col not in df.columns]
add_validation_check(
    'Required columns present',
    len(missing_required) == 0,
    'All required columns available' if len(missing_required) == 0 else f"Missing: {missing_required}"
)

add_validation_check(
    'No missing values',
    df.isnull().sum().sum() == 0,
    f"Remaining missing values: {int(df.isnull().sum().sum())}"
)

add_validation_check(
    'No duplicate rows',
    df.duplicated().sum() == 0,
    f"Remaining duplicates: {int(df.duplicated().sum())}"
)

add_validation_check(
    'Week converted to datetime',
    'Week' not in df.columns or pd.api.types.is_datetime64_any_dtype(df['Week']),
    'Week column is datetime64[ns]'
)

if 'Week' in df.columns:
    add_validation_check(
        'Valid Week values',
        df['Week'].isna().sum() == 0,
        f"Invalid Week values after parsing: {int(df['Week'].isna().sum())}"
    )

non_negative_columns = [
    col for col in [
        'Sales_Units', 'Sales_Value', 'MRP', 'Net_Price', 'Trade_Spend',
        'TV_Impressions', 'YouTube_Impressions', 'Facebook_Impressions',
        'Instagram_Impressions', 'Print_Readership', 'Radio_Listenership',
        'Weighted_Distribution', 'Numeric_Distribution', 'TDP', 'NOS'
    ] if col in df.columns
]
negative_counts = {col: int((df[col] < 0).sum()) for col in non_negative_columns}
negative_issues = {col: count for col, count in negative_counts.items() if count > 0}
add_validation_check(
    'Non-negative business metrics',
    len(negative_issues) == 0,
    'All monitored columns are non-negative' if len(negative_issues) == 0 else str(negative_issues)
)

flag_validation_cols = [col for col in flag_cols if col in df.columns]
invalid_flags = {}
for col in flag_validation_cols:
    if not set(df[col].dropna().unique()) <= {0, 1}:
        invalid_flags[col] = sorted(set(df[col].dropna().unique()) - {0, 1})
add_validation_check(
    'Binary flag values',
    len(invalid_flags) == 0,
    'All flag columns contain only 0/1' if len(invalid_flags) == 0 else str(invalid_flags)
)

if {'Sales_Units', 'Net_Price', 'Sales_Value'}.issubset(df.columns):
    expected_sales_value = df['Sales_Units'] * df['Net_Price']
    sales_value_match = np.isclose(df['Sales_Value'], expected_sales_value, rtol=0.02, atol=1.0)
    add_validation_check(
        'Sales value consistency',
        sales_value_match.mean() >= 0.95,
        f"Rows within tolerance: {sales_value_match.mean() * 100:.2f}%"
    )

if {'MRP', 'Net_Price'}.issubset(df.columns):
    price_rule_violations = int((df['Net_Price'] > df['MRP']).sum())
    add_validation_check(
        'Net price does not exceed MRP',
        price_rule_violations == 0,
        f"Violations found: {price_rule_violations}"
    )

validation_report = pd.DataFrame(validation_results)
print(validation_report.to_string(index=False))

all_checks_passed = validation_report['Passed'].all()
print(f"\nOverall Validation Status: {'PASSED' if all_checks_passed else 'FAILED'}")

STEP 5: DATA VALIDATION CHECKS
                        Check  Passed                                Details
     Required columns present    True         All required columns available
            No missing values    True            Remaining missing values: 0
            No duplicate rows    True                Remaining duplicates: 0
   Week converted to datetime   False          Week column is datetime64[ns]
            Valid Week values    True   Invalid Week values after parsing: 0
Non-negative business metrics    True All monitored columns are non-negative
           Binary flag values    True      All flag columns contain only 0/1
      Sales value consistency   False          Rows within tolerance: 93.68%
Net price does not exceed MRP    True                    Violations found: 0

Overall Validation Status: FAILED


## 6. Save Cleaned Dataset
Write the cleaned data to the processed folder.

In [10]:
print("="*80)
print("STEP 6: SAVE CLEANED DATASET")
print("="*80)

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'cleaned_data.csv'
df.to_csv(output_path, index=False)

print(f"✓ Cleaned dataset saved to: {output_path}")
print(f"Final Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Final Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nCleaned dataset preview:")
print(df.head().to_string(index=False))

STEP 6: SAVE CLEANED DATASET
✓ Cleaned dataset saved to: ..\data\processed\cleaned_data.csv
Final Dataset Shape: 11,232 rows × 28 columns
Final Memory Usage: 3.78 MB

Cleaned dataset preview:
      Week     Geo  Brand         SKU  Sales_Units  Sales_Value        MRP  Net_Price  Feature_Flag  Display_Flag  TPR_Flag  Trade_Spend  TV_Impressions  YouTube_Impressions  Facebook_Impressions  Instagram_Impressions  Print_Readership  Radio_Listenership  FB_Banner_Content_Score  IG_Banner_Content_Score  Weighted_Distribution  Numeric_Distribution       TDP      NOS        CPI  GDP_Growth  Festival_Index  Rainfall_Index
2022-07-04 CENTRAL BrandA BrandA_SKU1   106.327942 10551.108398  99.231750  99.231750         False         False     False 12108.946289    1.097592e+06        526732.437500          476584.68750          153004.953125      78906.937500       795667.220703                 74.71402                77.999977               0.326712              0.406848 36.923183 3.071753 120.954666 